In [1]:
import pandas as pd
import numpy as np

import os
import sys
from pathlib import Path
from bs4 import BeautifulSoup

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

from src.utils import limpiar_df, normalizar
from src.transfermarkt import parsear_plantilla

pd.set_option('display.max_columns', None)

In [2]:
ruta = '../data/raw/fbref_big5'

In [3]:
RAW = Path(ruta)

# FBref

In [4]:
for carpeta in sorted(RAW.iterdir()):
    if not carpeta.is_dir():
        continue
    temporada = carpeta.name          # '2021-2022'
    for archivo in sorted(carpeta.glob("*.parquet")):
        print(temporada, archivo.stem)

2021-2022 big5_2021-2022_misc
2021-2022 big5_2021-2022_playingtime
2021-2022 big5_2021-2022_shooting
2021-2022 big5_2021-2022_stats
2022-2023 big5_2022-2023_misc
2022-2023 big5_2022-2023_playingtime
2022-2023 big5_2022-2023_shooting
2022-2023 big5_2022-2023_stats
2023-2024 big5_2023-2024_misc
2023-2024 big5_2023-2024_playingtime
2023-2024 big5_2023-2024_shooting
2023-2024 big5_2023-2024_stats
2024-2025 big5_2024-2025_misc
2024-2025 big5_2024-2025_playingtime
2024-2025 big5_2024-2025_shooting
2024-2025 big5_2024-2025_stats
2025-2026 big5_2025-2026_misc
2025-2026 big5_2025-2026_playingtime
2025-2026 big5_2025-2026_shooting
2025-2026 big5_2025-2026_stats


In [5]:
por_temporada = []
CLAVE = ['Player', 'Squad', 'Comp', 'Born']

for carpeta in sorted(RAW.iterdir()):
    if not carpeta.is_dir():
        continue
    temporada = carpeta.name

    # 1. cargar las cuatro tablas de esta temporada
    tablas = {}
    for archivo in carpeta.glob("*.parquet"):
        tipo = archivo.stem.split("_")[-1]      # 'stats', 'misc', ...
        tablas[tipo] = limpiar_df(pd.read_parquet(archivo))
    # 2. unirlas entre sí  -> df_temporada
    # ...  aquí va tu lógica de merge
    

    base = tablas['stats'].copy()

    for tipo in ['shooting', 'misc', 'playingtime']:
        otra = tablas[tipo]
        nuevas = [c for c in otra.columns if c not in base.columns]
        base = base.merge(
            otra[CLAVE + nuevas],
            on=CLAVE,
            how='left',
            validate='one_to_one'
        )

    df_temporada = base

    df_temporada["Season"] = temporada
    por_temporada.append(df_temporada)

# 3. apilar todas las temporadas
completo = pd.concat(por_temporada, ignore_index=True)

In [6]:
completo.head()

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,Playing Time_MP,Playing Time_Starts,Playing Time_Min,Playing Time_90s,Performance_Gls,Performance_Ast,Performance_G+A,Performance_G-PK,Performance_PK,Performance_PKatt,Performance_CrdY,Performance_CrdR,Per 90 Minutes_Gls,Per 90 Minutes_Ast,Per 90 Minutes_G+A,Per 90 Minutes_G-PK,Per 90 Minutes_G+A-PK,90s,Standard_Gls,Standard_Sh,Standard_SoT,Standard_SoT%,Standard_Sh/90,Standard_SoT/90,Standard_G/Sh,Standard_G/SoT,Standard_PK,Standard_PKatt,Performance_2CrdY,Performance_Fls,Performance_Fld,Performance_Off,Performance_Crs,Performance_Int,Performance_TklW,Performance_PKwon,Performance_PKcon,Performance_OG,Playing Time_Mn/MP,Playing Time_Min%,Starts_Starts,Starts_Mn/Start,Starts_Compl,Subs_Subs,Subs_Mn/Sub,Subs_unSub,Team Success_PPM,Team Success_onG,Team Success_onGA,Team Success_+/-,Team Success_+/-90,Team Success_On-Off,Season
0,1,Max Aarons,ENG,DF,Norwich City,Premier League,21,2000,34,32,2881,32.0,0,2,2,0,0,0,8,0,0.00,0.06,0.06,0.00,0.06,32.0,0,13,2,15.4,0.41,0.06,0.00,0.00,0,0,0,26,52,1,50,26,45,NaN,NaN,1,85,84.2,32,89,29,2,22,2,0.53,18,69,-51,-1.59,+0.08,2021-2022
1,2,Yunis Abdelhamid,MAR,DF,Reims,Ligue 1,33,1987,34,34,2983,33.1,2,0,2,2,0,0,5,1,0.06,0.00,0.06,0.06,0.06,33.1,2,18,6,33.3,0.54,0.18,0.11,0.33,0,0,0,38,25,0,4,67,28,NaN,NaN,0,88,87.2,34,88,33,0,NaN,0,1.12,38,41,-3,-0.09,-0.50,2021-2022
2,3,Salis Abdul Samed,GHA,MF,Clermont Foot,Ligue 1,21,2000,31,29,2462,27.4,1,0,1,1,0,0,12,3,0.04,0.00,0.04,0.04,0.04,27.4,1,18,5,27.8,0.66,0.18,0.06,0.20,0,0,2,43,37,0,14,40,26,NaN,NaN,0,79,72.0,29,82,19,2,38,1,1.10,26,43,-17,-0.62,+0.69,2021-2022
3,4,Laurent Abergel,FRA,MF,Lorient,Ligue 1,28,1993,34,34,2956,32.8,0,2,2,0,0,0,9,0,0.00,0.06,0.06,0.00,0.06,32.8,0,30,7,23.3,0.91,0.21,0.00,0.00,0,0,0,41,69,1,37,54,63,NaN,NaN,0,87,86.4,34,87,26,0,NaN,0,0.82,27,59,-32,-0.97,-1.75,2021-2022
4,5,Charles Abi,FRA,FW,Saint-Étienne,Ligue 1,21,2000,1,1,45,0.5,0,0,0,0,0,0,0,0,0.00,0.00,0.00,0.00,0.00,0.5,0,0,0,NaN,0.00,0.00,NaN,NaN,0,0,0,1,0,0,1,0,0,NaN,NaN,0,45,1.3,1,45,0,0,NaN,0,1.00,0,0,0,0.00,+0.93,2021-2022


Corrigiendo tipos de dato

In [7]:
completo.drop(columns = ['Performance_PKwon', 'Performance_PKcon'], inplace = True)
completo['Team Success_+/-'] = completo['Team Success_+/-'].str.replace('+', '', regex = False)
completo['Team Success_+/-90'] = completo['Team Success_+/-90'].str.replace('+', '', regex = False)
completo['Team Success_On-Off'] = completo['Team Success_On-Off'].str.replace('+', '', regex = False)
vars_numericas = ['Age', 'Born', 'Playing Time_MP', 'Playing Time_Starts', 'Playing Time_Min', 'Playing Time_90s', 'Performance_Gls', 'Performance_Ast', 'Performance_G+A',
                  'Performance_G-PK', 'Performance_PK', 'Performance_PKatt', 'Performance_CrdY', 'Performance_CrdR', 'Per 90 Minutes_Gls', 'Per 90 Minutes_Ast', 'Per 90 Minutes_G+A', 'Per 90 Minutes_G+A-PK',
                  '90s', 'Standard_Gls', 'Standard_Sh', 'Standard_SoT', 'Standard_SoT%', 'Standard_Sh/90', 'Standard_SoT/90', 'Standard_G/Sh', 'Standard_G/SoT', 'Standard_PK', 'Standard_PKatt',
                  'Performance_2CrdY', 'Performance_Fls', 'Performance_Fld', 'Performance_Off', 'Performance_Crs', 'Performance_Int', 'Performance_TklW', 'Performance_OG',
                  'Playing Time_Mn/MP', 'Playing Time_Min%', 'Starts_Starts', 'Starts_Mn/Start', 'Starts_Compl', 'Subs_Subs', 'Subs_Mn/Sub', 'Subs_unSub',
                  'Team Success_PPM', 'Team Success_onG', 'Team Success_onGA', 'Team Success_+/-', 'Team Success_+/-90', 'Team Success_On-Off']
completo[vars_numericas] = completo[vars_numericas].apply(pd.to_numeric, errors='coerce')

In [8]:
# completo["Season_tm"] = completo["Season"].str.split("-").str[0].astype(int)

In [9]:
completo.head()

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,Playing Time_MP,Playing Time_Starts,Playing Time_Min,Playing Time_90s,Performance_Gls,Performance_Ast,Performance_G+A,Performance_G-PK,Performance_PK,Performance_PKatt,Performance_CrdY,Performance_CrdR,Per 90 Minutes_Gls,Per 90 Minutes_Ast,Per 90 Minutes_G+A,Per 90 Minutes_G-PK,Per 90 Minutes_G+A-PK,90s,Standard_Gls,Standard_Sh,Standard_SoT,Standard_SoT%,Standard_Sh/90,Standard_SoT/90,Standard_G/Sh,Standard_G/SoT,Standard_PK,Standard_PKatt,Performance_2CrdY,Performance_Fls,Performance_Fld,Performance_Off,Performance_Crs,Performance_Int,Performance_TklW,Performance_OG,Playing Time_Mn/MP,Playing Time_Min%,Starts_Starts,Starts_Mn/Start,Starts_Compl,Subs_Subs,Subs_Mn/Sub,Subs_unSub,Team Success_PPM,Team Success_onG,Team Success_onGA,Team Success_+/-,Team Success_+/-90,Team Success_On-Off,Season
0,1,Max Aarons,ENG,DF,Norwich City,Premier League,21.0,2000.0,34,32,2881,32.0,0,2,2,0,0,0,8,0,0.00,0.06,0.06,0.00,0.06,32.0,0,13,2,15.4,0.41,0.06,0.00,0.00,0,0,0,26,52,1,50,26,45,1,85,84.2,32,89.0,29,2,22.0,2,0.53,18,69,-51,-1.59,0.08,2021-2022
1,2,Yunis Abdelhamid,MAR,DF,Reims,Ligue 1,33.0,1987.0,34,34,2983,33.1,2,0,2,2,0,0,5,1,0.06,0.00,0.06,0.06,0.06,33.1,2,18,6,33.3,0.54,0.18,0.11,0.33,0,0,0,38,25,0,4,67,28,0,88,87.2,34,88.0,33,0,NaN,0,1.12,38,41,-3,-0.09,-0.50,2021-2022
2,3,Salis Abdul Samed,GHA,MF,Clermont Foot,Ligue 1,21.0,2000.0,31,29,2462,27.4,1,0,1,1,0,0,12,3,0.04,0.00,0.04,0.04,0.04,27.4,1,18,5,27.8,0.66,0.18,0.06,0.20,0,0,2,43,37,0,14,40,26,0,79,72.0,29,82.0,19,2,38.0,1,1.10,26,43,-17,-0.62,0.69,2021-2022
3,4,Laurent Abergel,FRA,MF,Lorient,Ligue 1,28.0,1993.0,34,34,2956,32.8,0,2,2,0,0,0,9,0,0.00,0.06,0.06,0.00,0.06,32.8,0,30,7,23.3,0.91,0.21,0.00,0.00,0,0,0,41,69,1,37,54,63,0,87,86.4,34,87.0,26,0,NaN,0,0.82,27,59,-32,-0.97,-1.75,2021-2022
4,5,Charles Abi,FRA,FW,Saint-Étienne,Ligue 1,21.0,2000.0,1,1,45,0.5,0,0,0,0,0,0,0,0,0.00,0.00,0.00,0.00,0.00,0.5,0,0,0,NaN,0.00,0.00,NaN,NaN,0,0,0,1,0,0,1,0,0,0,45,1.3,1,45.0,0,0,NaN,0,1.00,0,0,0,0.00,0.93,2021-2022


# Transfermarkt

In [10]:
HTML = ROOT / "data/raw/transfermarkt/plantillas_html"

In [11]:
from src.transfermarkt import parsear_todas
tm = parsear_todas(HTML)
print(tm.shape)
print(tm['valor_eur'].isna().mean())
tm.to_parquet(ROOT / "data/interim/transfermarkt.parquet")

(19014, 14)
0.056800252445566426


In [12]:
clubes = pd.read_csv(ROOT / "data/raw/transfermarkt/clubes.csv")
clubes['club_id'] = clubes['club_id'].astype(str)
tm['club_id'] = tm['club_id'].astype(str)

tm = tm.merge(
    clubes[['club_id', 'saison_id', 'club', 'liga']],
    on=['club_id', 'saison_id'],
    how='left'
)
print(tm['club'].isna().mean())

0.0


In [13]:
tm.head()

,player_id,player_tm,dorsal,posicion_tm,nacimiento,edad_tm,nacionalidad,altura_m,pie,fecha_fichaje,valor_eur,liga_codigo,saison_id,club_id,club,liga
0,502676,Giorgi Mamardashvili,28,Goalkeeper,29/09/2000,21,Georgia,1.97,left,28/08/2024,10000000.0,ES1,2021,1049,Valencia CF,La Liga
1,146227,Jasper Cillessen,13,Goalkeeper,22/04/1989,33,Netherlands,1.87,right,01/07/2019,3000000.0,ES1,2021,1049,Valencia CF,La Liga
2,227805,Jaume Doménech,1,Goalkeeper,05/11/1990,31,Spain,1.85,right,01/07/2015,2000000.0,ES1,2021,1049,Valencia CF,La Liga
3,398131,Cristian Rivero,25,Goalkeeper,21/03/1998,24,Spain,1.88,right,01/08/2020,200000.0,ES1,2021,1049,Valencia CF,La Liga
4,707271,Charlie Pérez,35,Goalkeeper,07/06/2002,20,Spain,NaN,right,,NaN,ES1,2021,1049,Valencia CF,La Liga


## Cruzar ambas tablas

Quitamos los registros duplicados de la misma temporada

In [14]:
tm[tm["player_tm"] == "Lucien Agoumé"]

,player_id,player_tm,dorsal,posicion_tm,nacimiento,edad_tm,nacionalidad,altura_m,pie,fecha_fichaje,valor_eur,liga_codigo,saison_id,club_id,club,liga
2060,569384,Lucien Agoumé,42,Defensive Midfield,09/02/2002,22,France,1.85,right,06/08/2024,6000000.0,ES1,2023,368,Sevilla FC,La Liga
2799,569384,Lucien Agoumé,18,Defensive Midfield,09/02/2002,23,France,1.85,right,06/08/2024,8000000.0,ES1,2024,368,Sevilla FC,La Liga
3619,569384,Lucien Agoumé,18,Defensive Midfield,09/02/2002,24,France,1.85,right,06/08/2024,15000000.0,ES1,2025,368,Sevilla FC,La Liga
4339,569384,Lucien Agoumé,10,Defensive Midfield,09/02/2002,20,France,1.85,right,31/08/2021,7000000.0,FR1,2021,3911,Stade Brestois 29,Ligue 1
4747,569384,Lucien Agoumé,8,Defensive Midfield,09/02/2002,21,France,1.85,right,01/09/2022,7000000.0,FR1,2022,1095,ESTAC Troyes,Ligue 1
12162,569384,Lucien Agoumé,-,Defensive Midfield,09/02/2002,20,France,1.85,right,05/07/2019,7000000.0,IT1,2021,46,Inter Milan,Serie A
13035,569384,Lucien Agoumé,-,Defensive Midfield,09/02/2002,21,France,1.85,right,05/07/2019,7000000.0,IT1,2022,46,Inter Milan,Serie A
13797,569384,Lucien Agoumé,42,Defensive Midfield,09/02/2002,22,France,1.85,right,05/07/2019,6000000.0,IT1,2023,46,Inter Milan,Serie A


In [15]:
chk = tm.groupby(['player_id','saison_id'])['valor_eur'].nunique()
print((chk > 1).mean())

0.0


In [16]:
tm_unico = tm.drop_duplicates(subset=['player_id', 'saison_id'])
print(tm.shape, '->', tm_unico.shape)

(19014, 16) -> (18002, 16)


In [17]:
tm_unico[tm_unico['player_tm'] == 'Lucien Agoumé'][['saison_id','club','valor_eur']]

,saison_id,club,valor_eur
2060,2023,Sevilla FC,6000000.0
2799,2024,Sevilla FC,8000000.0
3619,2025,Sevilla FC,15000000.0
4339,2021,Stade Brestois 29,7000000.0
4747,2022,ESTAC Troyes,7000000.0


## Cruzamos los jugadores entre ambos datasets

In [18]:
# Mapeo FBref<->Transfermarkt de worldfootballR_data (JaseZiv). Se descargó una sola vez de
# https://raw.githubusercontent.com/JaseZiv/worldfootballR_data/master/raw-data/fbref-tm-player-mapping/output/fbref_to_tm_mapping.csv
# y quedó congelado en data/raw/ como el resto de las fuentes -- así este notebook no depende
# de tener red ni de que ese archivo cambie en la rama master del repo original.
mapping = pd.read_csv(ROOT / "data/raw/fbref_tm_mapping/fbref_to_tm_mapping.csv", encoding="latin-1")
print(mapping.shape)
print(list(mapping.columns))
mapping.head()

(15440, 4)
['PlayerFBref', 'UrlFBref', 'UrlTmarkt', 'TmPos']


,PlayerFBref,UrlFBref,UrlTmarkt,TmPos
0,A.J. DeLaGarza,https://fbref.com/en/players/171b3c37/AJ-DeLaG...,https://www.transfermarkt.com/a-j-delagarza/pr...,Right-Back
1,AJ Marcucci,https://fbref.com/en/players/20c86a38/AJ-Marcucci,https://www.transfermarkt.com/aj-marcucci/prof...,Goalkeeper
2,Aapo Halme,https://fbref.com/en/players/02b952ce/Aapo-Halme,https://www.transfermarkt.com/aapo-halme/profi...,Centre-Back
3,Aaron Bastiaans,https://fbref.com/en/players/fb979733/Aaron-Ba...,https://www.transfermarkt.com/aaron-bastiaans/...,Left Winger
4,Aaron Ciammaglichella,https://fbref.com/en/players/20525799/Aaron-Ci...,https://www.transfermarkt.com/aaron-ciammaglic...,Central Midfield


In [19]:
# --- preparar llaves ---
mapping['player_id'] = mapping['UrlTmarkt'].str.extract(r'/spieler/(\d+)')
mapping['nom'] = mapping['PlayerFBref'].map(normalizar)
mapping_u = mapping.dropna(subset=['player_id']).drop_duplicates(subset=['nom'])

fb = completo.copy()
fb['nom'] = fb['Player'].map(normalizar)
fb['saison_id'] = fb['Season'].str.split('-').str[0].astype(int)

tmu = tm.drop_duplicates(subset=['player_id','saison_id']).copy()
tmu['player_id'] = tmu['player_id'].astype(str)

In [20]:
# --- paso 1: por mapping ---
fb = fb.merge(mapping_u[['nom','player_id']], on='nom', how='left')
print('con player_id:', fb['player_id'].notna().mean())

con player_id: 0.7854555586514349


In [21]:
# --- paso 2: fallback por nombre + año de nacimiento ---
tmu['anio_nac'] = tmu['nacimiento'].str.split('/').str[-1].astype(float)
tmu['nom'] = tmu['player_tm'].map(normalizar)
fallback = tmu.drop_duplicates(subset=['nom','anio_nac'])[['nom','anio_nac','player_id']]
fallback = fallback.rename(columns={'player_id':'pid_fb'})

fb = fb.merge(fallback, left_on=['nom','Born'], right_on=['nom','anio_nac'], how='left')
fb['player_id'] = fb['player_id'].fillna(fb['pid_fb'])
print('tras fallback:', fb['player_id'].notna().mean())

tras fallback: 0.9783365840066871


In [22]:
# --- paso 3: traer el valor ---
dataset = fb.merge(
    tmu[['player_id','saison_id','valor_eur','posicion_tm','altura_m','nacimiento','club']],
    on=['player_id','saison_id'], how='left', validate='many_to_one'
)
print('con valor:', dataset['valor_eur'].notna().mean())

con valor: 0.9637085539147395


In [23]:
chk = dataset.dropna(subset=['club'])
coincide = (chk['Squad'].map(normalizar) == chk['club'].map(normalizar))
print(coincide.mean())
print(chk.loc[~coincide, ['Player','Squad','club','Season']].head(20))

0.1254927255787286
                  Player           Squad                   club     Season
1       Yunis Abdelhamid           Reims            Stade Reims  2021-2022
3        Laurent Abergel         Lorient             FC Lorient  2021-2022
4            Charles Abi   Saint-Étienne       AS Saint-Étienne  2021-2022
5         Dickson Abiama  Greuther Fürth   SpVgg Greuther Fürth  2021-2022
6         Matthis Abline          Rennes       Stade Rennais FC  2021-2022
7          Tammy Abraham            Roma             Chelsea FC  2021-2022
8             Luis Abram         Granada             Granada CF  2021-2022
9       Francesco Acerbi           Lazio               SS Lazio  2021-2022
10           Ragnar Ache       Frankfurt    Eintracht Frankfurt  2021-2022
11          Mohamed Achi          Nantes              FC Nantes  2021-2022
12          Marcos Acuña         Sevilla             Sevilla FC  2021-2022
13             Ché Adams     Southampton         Southampton FC  2021-2022
15    

In [24]:
def clave_club(s):
    n = normalizar(s)
    quitar = {'fc','cf','sc','ac','as','ss','sv','vfl','vfb','tsg','spvgg',
              'stade','deportivo','club','athletic','real','sd','rc','us','ogc'}
    return ' '.join(w for w in n.split() if w not in quitar)

chk2 = chk.copy()
chk2['ok'] = chk2['Squad'].map(clave_club) == chk2['club'].map(clave_club)
print(chk2['ok'].mean())
print(chk2.loc[~chk2['ok'], ['Player','Squad','club','Season']].sample(20))

0.5599512649609403
                   Player         Squad                      club     Season
2254          Sergio Rico      Mallorca              RCD Mallorca  2021-2022
11770     Sheraldo Becker      Mainz 05                CA Osasuna  2025-2026
7814       Patrick Ouotro    Strasbourg      RC Strasbourg Alsace  2023-2024
3260     Antonin Bobichon        Angers                Angers SCO  2022-2023
2277            Joe Rodon     Tottenham         Tottenham Hotspur  2021-2022
11297     Adrien Truffert        Rennes          Stade Rennais FC  2024-2025
11658       Carlens Arcus        Angers                Angers SCO  2025-2026
12707    Omari Hutchinson    Nottingham         Nottingham Forest  2025-2026
3846        Wesley Fofana       Chelsea            Leicester City  2022-2023
10270    Javier Manquillo    Celta Vigo             Celta de Vigo  2024-2025
11634      Benjamin André         Lille                LOSC Lille  2025-2026
7313          David López        Girona              RCD 

In [25]:
chk.head()

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,Playing Time_MP,Playing Time_Starts,Playing Time_Min,Playing Time_90s,Performance_Gls,Performance_Ast,Performance_G+A,Performance_G-PK,Performance_PK,Performance_PKatt,Performance_CrdY,Performance_CrdR,Per 90 Minutes_Gls,Per 90 Minutes_Ast,Per 90 Minutes_G+A,Per 90 Minutes_G-PK,Per 90 Minutes_G+A-PK,90s,Standard_Gls,Standard_Sh,Standard_SoT,Standard_SoT%,Standard_Sh/90,Standard_SoT/90,Standard_G/Sh,Standard_G/SoT,Standard_PK,Standard_PKatt,Performance_2CrdY,Performance_Fls,Performance_Fld,Performance_Off,Performance_Crs,Performance_Int,Performance_TklW,Performance_OG,Playing Time_Mn/MP,Playing Time_Min%,Starts_Starts,Starts_Mn/Start,Starts_Compl,Subs_Subs,Subs_Mn/Sub,Subs_unSub,Team Success_PPM,Team Success_onG,Team Success_onGA,Team Success_+/-,Team Success_+/-90,Team Success_On-Off,Season,nom,saison_id,player_id,anio_nac,pid_fb,valor_eur,posicion_tm,altura_m,nacimiento,club
0,1,Max Aarons,ENG,DF,Norwich City,Premier League,21.0,2000.0,34,32,2881,32.0,0,2,2,0,0,0,8,0,0.00,0.06,0.06,0.00,0.06,32.0,0,13,2,15.4,0.41,0.06,0.00,0.00,0,0,0,26,52,1,50,26,45,1,85,84.2,32,89.0,29,2,22.0,2,0.53,18,69,-51,-1.59,0.08,2021-2022,max aarons,2021,471690,2000.0,471690,22000000.0,Right-Back,1.77,04/01/2000,Norwich City
1,2,Yunis Abdelhamid,MAR,DF,Reims,Ligue 1,33.0,1987.0,34,34,2983,33.1,2,0,2,2,0,0,5,1,0.06,0.00,0.06,0.06,0.06,33.1,2,18,6,33.3,0.54,0.18,0.11,0.33,0,0,0,38,25,0,4,67,28,0,88,87.2,34,88.0,33,0,NaN,0,1.12,38,41,-3,-0.09,-0.50,2021-2022,yunis abdelhamid,2021,199704,1987.0,199704,1200000.0,Centre-Back,1.90,28/09/1987,Stade Reims
2,3,Salis Abdul Samed,GHA,MF,Clermont Foot,Ligue 1,21.0,2000.0,31,29,2462,27.4,1,0,1,1,0,0,12,3,0.04,0.00,0.04,0.04,0.04,27.4,1,18,5,27.8,0.66,0.18,0.06,0.20,0,0,2,43,37,0,14,40,26,0,79,72.0,29,82.0,19,2,38.0,1,1.10,26,43,-17,-0.62,0.69,2021-2022,salis abdul samed,2021,688707,2000.0,688707,3000000.0,Defensive Midfield,1.79,26/03/2000,Clermont Foot 63
3,4,Laurent Abergel,FRA,MF,Lorient,Ligue 1,28.0,1993.0,34,34,2956,32.8,0,2,2,0,0,0,9,0,0.00,0.06,0.06,0.00,0.06,32.8,0,30,7,23.3,0.91,0.21,0.00,0.00,0,0,0,41,69,1,37,54,63,0,87,86.4,34,87.0,26,0,NaN,0,0.82,27,59,-32,-0.97,-1.75,2021-2022,laurent abergel,2021,238626,1993.0,238626,3000000.0,Defensive Midfield,1.70,01/02/1993,FC Lorient
4,5,Charles Abi,FRA,FW,Saint-Étienne,Ligue 1,21.0,2000.0,1,1,45,0.5,0,0,0,0,0,0,0,0,0.00,0.00,0.00,0.00,0.00,0.5,0,0,0,NaN,0.00,0.00,NaN,NaN,0,0,0,1,0,0,1,0,0,0,45,1.3,1,45.0,0,0,NaN,0,1.00,0,0,0,0.00,0.93,2021-2022,charles abi,2021,418658,2000.0,418658,1500000.0,Centre-Forward,1.89,12/04/2000,AS Saint-Étienne


In [26]:
chk[chk["Player"] == "Lucien Agoume"]

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,Playing Time_MP,Playing Time_Starts,Playing Time_Min,Playing Time_90s,Performance_Gls,Performance_Ast,Performance_G+A,Performance_G-PK,Performance_PK,Performance_PKatt,Performance_CrdY,Performance_CrdR,Per 90 Minutes_Gls,Per 90 Minutes_Ast,Per 90 Minutes_G+A,Per 90 Minutes_G-PK,Per 90 Minutes_G+A-PK,90s,Standard_Gls,Standard_Sh,Standard_SoT,Standard_SoT%,Standard_Sh/90,Standard_SoT/90,Standard_G/Sh,Standard_G/SoT,Standard_PK,Standard_PKatt,Performance_2CrdY,Performance_Fls,Performance_Fld,Performance_Off,Performance_Crs,Performance_Int,Performance_TklW,Performance_OG,Playing Time_Mn/MP,Playing Time_Min%,Starts_Starts,Starts_Mn/Start,Starts_Compl,Subs_Subs,Subs_Mn/Sub,Subs_unSub,Team Success_PPM,Team Success_onG,Team Success_onGA,Team Success_+/-,Team Success_+/-90,Team Success_On-Off,Season,nom,saison_id,player_id,anio_nac,pid_fb,valor_eur,posicion_tm,altura_m,nacimiento,club
23,24,Lucien Agoume,FRA,MF,Brest,Ligue 1,19.0,2002.0,27,21,1868,20.8,0,0,0,0,0,0,6,0,0.00,0.00,0.00,0.00,0.00,20.8,0,15,1,6.7,0.72,0.05,0.00,0.0,0,0,0,31,15,0,6,49,33,0,69,54.6,21,83.0,13,6,21.0,4,1.30,27,32,-5,-0.24,-0.07,2021-2022,lucien agoume,2021,569384,2002.0,569384,7000000.0,Defensive Midfield,1.85,09/02/2002,Stade Brestois 29
2953,32,Lucien Agoume,FRA,MF,Troyes,Ligue 1,20.0,2002.0,15,14,1094,12.2,0,1,1,0,0,0,5,0,0.00,0.08,0.08,0.00,0.08,12.2,0,9,3,33.3,0.74,0.25,0.00,0.0,0,0,0,30,8,1,9,25,23,0,73,32.0,14,78.0,8,1,9.0,6,0.40,9,29,-20,-1.65,-1.03,2022-2023,lucien agoume,2022,569384,2002.0,569384,7000000.0,Defensive Midfield,1.85,09/02/2002,ESTAC Troyes
5843,33,Lucien Agoume,FRA,MF,Sevilla,La Liga,21.0,2002.0,12,9,777,8.6,0,0,0,0,0,0,3,0,0.00,0.00,0.00,0.00,0.00,8.6,0,6,2,33.3,0.69,0.23,0.00,0.0,0,0,0,19,2,0,3,7,16,0,65,22.7,9,79.0,5,3,22.0,1,1.42,11,12,-1,-0.12,0.05,2023-2024,lucien agoume,2023,569384,2002.0,569384,6000000.0,Defensive Midfield,1.85,09/02/2002,Sevilla FC
5844,34,Lucien Agoume,FRA,MF,Inter,Serie A,21.0,2002.0,1,0,5,0.1,0,0,0,0,0,0,0,0,0.00,0.00,0.00,0.00,0.00,0.1,0,0,0,NaN,0.00,0.00,NaN,NaN,0,0,0,0,0,0,0,0,0,0,5,0.1,0,NaN,0,1,5.0,16,3.00,1,0,1,18.00,16.26,2023-2024,lucien agoume,2023,569384,2002.0,569384,6000000.0,Defensive Midfield,1.85,09/02/2002,Sevilla FC
8705,43,Lucien Agoume,FRA,MF,Sevilla,La Liga,22.0,2002.0,35,24,2114,23.5,1,3,4,1,0,0,5,1,0.04,0.13,0.17,0.04,0.17,23.5,1,10,2,20.0,0.43,0.09,0.10,0.5,0,0,0,49,6,2,17,40,32,0,60,61.8,24,84.0,14,11,10.0,1,1.09,26,39,-13,-0.55,-0.55,2024-2025,lucien agoume,2024,569384,2002.0,569384,8000000.0,Defensive Midfield,1.85,09/02/2002,Sevilla FC
11558,42,Lucien Agoume,FRA,MF,Sevilla,La Liga,23.0,2002.0,34,31,2686,29.8,1,3,4,1,0,0,12,0,0.03,0.10,0.13,0.03,0.13,29.8,1,15,2,13.3,0.50,0.07,0.07,0.5,0,0,0,58,16,8,18,49,40,0,79,78.5,31,84.0,20,3,26.0,1,1.18,39,48,-9,-0.30,0.31,2025-2026,lucien agoume,2025,569384,2002.0,569384,15000000.0,Defensive Midfield,1.85,09/02/2002,Sevilla FC


In [27]:
chk.shape

(13953, 69)

## Vencimiento de contrato (2021-22 a 2023-24, con datos de Kaggle)

FBref y Transfermarkt no traen fecha de vencimiento de contrato, así que la sacamos de otro lado: el dataset de Kaggle "EA Sports FC 24". Cubre hasta 2023-2024 nada más -- las últimas dos temporadas las resolvemos más abajo con Wayback Machine. El cruce es por nombre + año de nacimiento, porque Kaggle guarda el nombre completo ("Erling Braut Haaland") y no calza tal cual con el de FBref.

In [28]:
KAGGLE_VERSION_A_SEASON = {22: "2021-2022", 23: "2022-2023", 24: "2023-2024"}


def clave_jugador(nombre):
    palabras = normalizar(nombre).split()
    if not palabras:
        return None
    return (palabras[0][0], palabras[-1])


kag = pd.read_csv(
    ROOT / "data/raw/kaggle_fifa/ea_sports_fc24_male_players.csv.zip",
    usecols=["fifa_version", "league_name", "long_name", "dob", "club_contract_valid_until_year"],
    low_memory=False,
)
kag = kag[kag["fifa_version"].isin(KAGGLE_VERSION_A_SEASON)].copy()
kag["Season"] = kag["fifa_version"].map(KAGGLE_VERSION_A_SEASON)
kag["Comp"] = kag["league_name"]
kag = kag[kag["Comp"].isin(["La Liga", "Premier League", "Serie A", "Bundesliga", "Ligue 1"])]
kag = kag.dropna(subset=["club_contract_valid_until_year"])

kag["clave"] = kag["long_name"].map(clave_jugador)
kag["dob"] = pd.to_datetime(kag["dob"])

dup = kag.duplicated(subset=["Comp", "Season", "clave"]).sum()
print(f"Duplicados por (Comp, Season, clave) en Kaggle: {dup} de {len(kag)}")
kag = kag.drop_duplicates(subset=["Comp", "Season", "clave"], keep="first")

Duplicados por (Comp, Season, clave) en Kaggle: 210 de 10766


In [29]:
chk["clave"] = chk["Player"].map(clave_jugador)

chk = chk.merge(
    kag[["Comp", "Season", "clave", "dob", "club_contract_valid_until_year"]],
    on=["Comp", "Season", "clave"], how="left", validate="many_to_one",
)
print(f"Filas con año de vencimiento (Kaggle): {chk['club_contract_valid_until_year'].notna().mean():.3f}")

# Contrato de fútbol europeo casi siempre vence el 30 de junio (fin de temporada) --
# Kaggle solo da el año, así que aproximamos el día así.
chk["contrato_vence"] = pd.to_datetime(dict(
    year=chk["club_contract_valid_until_year"].dropna().astype(int), month=6, day=30
))

# Validación: si el año de nacimiento de Kaggle no coincide (+/-1) con el de FBref,
# probable colisión de (inicial, apellido) -- se anula el match en vez de dejarlo mal.
sospechoso = (chk["Born"] - chk["dob"].dt.year).abs() > 1
sospechoso = sospechoso.fillna(False) & chk["contrato_vence"].notna()
print(f"Matches sospechosos anulados (año de nacimiento no coincide): {sospechoso.sum()}")
chk.loc[sospechoso, "contrato_vence"] = pd.NaT

chk = chk.drop(columns=["clave", "dob", "club_contract_valid_until_year"])

Filas con año de vencimiento (Kaggle): 0.348
Matches sospechosos anulados (año de nacimiento no coincide): 86


## Vencimiento de contrato (2024-25 y 2025-26, con Wayback Machine)

Para las temporadas que Kaggle no cubre, buscamos el "Contrato hasta" en capturas viejas de Transfermarkt guardadas en Wayback Machine -- no hace falta pedirle nada nuevo a Transfermarkt, ya teníamos los HTML descargados. De los ~5,200 pares jugador-temporada, un poco más de la mitad terminó con contrato encontrado; el resto son huecos reales de las fuentes, no errores nuestros.

In [30]:
way = pd.read_csv(ROOT / "data/raw/wayback_contratos/contratos.csv")
way["player_id"] = way["player_id"].astype(str)
way["contrato_vence"] = pd.to_datetime(way["contrato_vence"])
way = way.rename(columns={"temporada": "Season"})
way = way.dropna(subset=["contrato_vence"])[["player_id", "Season", "contrato_vence"]]

print(f"Contratos de Wayback a incorporar: {len(way)}")
dup = way.duplicated(subset=["player_id", "Season"]).sum()
print(f"Duplicados (player_id, Season): {dup}")

Contratos de Wayback a incorporar: 2951
Duplicados (player_id, Season): 0


In [31]:
chk = chk.merge(way, on=["player_id", "Season"], how="left", suffixes=("", "_way"))
chk["contrato_vence"] = chk["contrato_vence"].fillna(chk["contrato_vence_way"])
chk = chk.drop(columns=["contrato_vence_way"])

print(f"Cobertura final de contrato_vence: {chk['contrato_vence'].notna().mean():.3f}")
chk.groupby("Season")["contrato_vence"].apply(lambda s: s.notna().mean())

Cobertura final de contrato_vence: 0.567


Season
2021-2022    0.516673
2022-2023    0.542674
2023-2024    0.638030
2024-2025    0.600573
2025-2026    0.539764
Name: contrato_vence, dtype: float64

In [32]:
chk.to_parquet(ROOT / "data/interim/fbref_tm_dataset.parquet", index = False)